# Auto ARIMA Benchmark — smooth 1.1.1 (2026-10-01)

Rerun of the auto-ARIMA comparison on M1 + M3 + Tourism (5,315 series) with
**smooth 1.1.1**, against the latest smooth run on disk, 2026-09-30
(`2026-09-30-benchmark-automsarima-master.csv`, smooth 1.0.9 after the
Hannan-Rissanen initialisation).

**Only the smooth rows were recomputed**: `AutoMSARIMA Back` and `AutoMSARIMA Opt`.
The competitor rows are not refitted: RMSSE and the scaled pinball come from
`smooth-paper/Data/python-arima-rmsse.csv` and `python-arima-pinball.csv`, and
their SAME from `2026-08-24-benchmark-arima-point.npy` (whose RMSSE matches the
CSV exactly). The previous smooth run stored RMSSE and time only, so there is no
SAME or pinball "before".

One fit per (method, series) gives the point forecast and the 99 quantiles
(`interval="prediction"`, `side="both"`).

**Timing caveat.** This run used 4 cores and Python 3.14, so times are not
comparable with the saved runs and are not discussed.

In [1]:
import os, glob, time, signal, datetime, subprocess, sys, warnings, multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

HERE = os.getcwd()
DATA = os.path.join(HERE, "smooth-paper", "Data")
LEVELS = np.round(np.arange(0.01, 1.0, 0.01), 2)               # 99 quantile levels
TWO = [round(1 - 2 * t, 2) for t in LEVELS if t < 0.5]          # two-sided interval levels
VERSION_TAG = "smooth-1-1-1"
TASK_TIMEOUT = int(os.environ.get("BENCH_TASK_TIMEOUT", "120"))
DATE = "2026-10-01"
KIND = "arima"

## Versions

In [2]:
from importlib.metadata import version as pkg_version

print(f"python {sys.version.split()[0]}   cores {multiprocessing.cpu_count()}")
for _p in ("smooth", "greybox", "numpy", "pandas", "fcompdata"):
    print(f"{_p:12s} v{pkg_version(_p)}")
print("\ngit HEAD:", subprocess.run(["git", "log", "-1", "--format=%h %s"],
                                  capture_output=True, text=True).stdout.strip())

python 3.14.4   cores 4
smooth       v1.1.1
greybox      v1.0.7
numpy        v2.5.3
pandas       v3.0.6
fcompdata    v0.1.2

git HEAD: 92f3961e Fix sign-compare warnings in adamCore::reapply loops


## Datasets

In [3]:
from fcompdata import M1, M3, Tourism

datasets = ([M1[k] for k in M1.keys()]
            + [M3[k] for k in M3.keys()]
            + [Tourism[k] for k in Tourism.keys()])
print(f"M1: {len(M1)}   M3: {len(M3)}   Tourism: {len(Tourism)}   total: {len(datasets)}")

M1: 1001   M3: 3003   Tourism: 1311   total: 5315


## Methods and harness

In [4]:
METHODS = [
    ("AutoMSARIMA Back", {"initial": "backcasting"}),
    ("AutoMSARIMA Opt", {"initial": "optimal"}),
]
SMOOTH = [name for name, _ in METHODS]


def fit_predict(cfg, s):
    from smooth import AutoMSARIMA
    m = AutoMSARIMA(lags=lags_of(s), initial=cfg["initial"])
    m.fit(np.asarray(s.x, float))
    return quantiles(m.predict(h=s.h, interval="prediction", level=TWO, side="both"))

In [5]:
def lags_of(s):
    return [1, s.period] if s.period > 1 else [1]


def quantiles(fc):
    # The 99 quantiles from the two-sided intervals, with the mean at 0.5.
    mean = np.asarray(fc.mean, float).ravel()
    lower, upper = np.asarray(fc.lower, float), np.asarray(fc.upper, float)
    lo = {round(float(c), 2): i for i, c in enumerate(fc.lower.columns)}
    up = {round(float(c), 2): i for i, c in enumerate(fc.upper.columns)}
    Q = np.empty((mean.size, 99))
    for k, t in enumerate(LEVELS):
        if abs(t - 0.5) < 1e-9:
            Q[:, k] = mean
        elif t < 0.5:
            Q[:, k] = lower[:, lo[round(t, 2)]]
        else:
            Q[:, k] = upper[:, up[round(t, 2)]]
    return mean, Q


def score(s, mean, Q):
    # RMSSE, SAME and the scaled pinball averaged over the 99 levels.
    x, y = np.asarray(s.x, float), np.asarray(s.xx, float)
    scale_sq, scale_abs = np.mean(np.diff(x) ** 2), np.mean(np.abs(np.diff(x)))
    if not np.all(np.isfinite(mean)):
        return np.nan, np.nan, np.nan
    rmsse = np.sqrt(np.mean((y - mean) ** 2) / scale_sq) if scale_sq > 0 else np.nan
    same = np.abs(np.mean(y - mean)) / scale_abs if scale_abs > 0 else np.nan
    pinball = np.nan
    if Q is not None and np.all(np.isfinite(Q)):
        diff = y[:, None] - np.sort(Q, axis=1)
        loss = np.maximum(LEVELS * diff, (LEVELS - 1) * diff).mean(axis=0)
        pinball = np.mean(loss) / (scale_abs if scale_abs > 0 else 1.0)
    return rmsse, same, pinball


def _alarm(signum, frame):
    raise TimeoutError("task timeout")


def task(args):
    # One (method, series) fit: returns rmsse, same, time, pinball.
    warnings.filterwarnings("ignore")
    j, i, s, cfg = args
    out = np.full(4, np.nan)
    try:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(TASK_TIMEOUT)
        start = time.time()
        mean, Q = fit_predict(cfg, s)
        out[2] = time.time() - start
        out[0], out[1], out[3] = score(s, mean[: s.h], None if Q is None else Q[: s.h])
    except Exception:
        pass
    finally:
        signal.alarm(0)
    return j, i, out


def run(methods, workers):
    result = np.full((len(methods), len(datasets), 4), np.nan)
    tasks = [(j, i, s, cfg) for j, (_, cfg) in enumerate(methods) for i, s in enumerate(datasets)]
    start = time.time()
    ctx = multiprocessing.get_context("fork")
    with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:
        for done, f in enumerate(as_completed([ex.submit(task, t) for t in tasks]), 1):
            j, i, out = f.result()
            result[j, i] = out
            if done % 2000 == 0:
                print(f"  {done}/{len(tasks)}  {(time.time() - start) / 60:.1f} min", flush=True)
    print(f"done in {(time.time() - start) / 60:.1f} min")
    return result

## Run or load

In [6]:
# Fit the smooth rows on the current build, or load the saved 1.1.1 array
# (BENCH_RERUN=1 forces a refit). Columns: rmsse, same, time, pinball.
saved_file = os.path.join(HERE, f"{DATE}-benchmark-{KIND}-{VERSION_TAG}.npy")
if os.environ.get("BENCH_RERUN", "0") == "1" or not os.path.exists(saved_file):
    workers = int(os.environ.get("BENCH_WORKERS", multiprocessing.cpu_count()))
    print(f"{len(METHODS)} methods x {len(datasets)} series, {workers} workers")
    current = run(METHODS, workers)
    np.save(saved_file, current)
else:
    current = np.load(saved_file)
print(os.path.basename(saved_file), current.shape)

2 methods x 5315 series, 4 workers


  2000/10630  6.1 min


  4000/10630  14.8 min


  6000/10630  33.2 min


  8000/10630  48.5 min


  10000/10630  91.9 min


done in 94.2 min
2026-10-01-benchmark-arima-smooth-1-1-1.npy (2, 5315, 4)


## Saved runs

In [7]:
# The current run, the previous smooth run and the competitors (saved files).
RESULT = {name: {"rmsse": current[j, :, 0], "same": current[j, :, 1],
                 "pinball": current[j, :, 3], "source": DATE}
          for j, name in enumerate(SMOOTH)}

prev = pd.read_csv("2026-09-30-benchmark-automsarima-master.csv")
PREVIOUS_LABEL = "2026-09-30 (smooth 1.0.9)"
PREVIOUS = {name: {"rmsse": prev[prev.method == name].sort_values("series").rmsse.to_numpy(),
                   "same": None, "pinball": None}
            for name in SMOOTH}

saved_names = [line.split("\t")[0] for line in
               open("2026-08-24-benchmark-arima-methods.txt").read().splitlines() if line]
same_saved = np.load("2026-08-24-benchmark-arima-point.npy")
rmsse_csv = pd.read_csv(os.path.join(DATA, "python-arima-rmsse.csv"))
pinball_csv = pd.read_csv(os.path.join(DATA, "python-arima-pinball.csv"))
for name in rmsse_csv.columns:
    RESULT[name] = {"rmsse": rmsse_csv[name].to_numpy(),
                    "same": same_saved[saved_names.index(name), :, 1],
                    "pinball": pinball_csv[name].to_numpy(),
                    "source": "csv (SAME: 2026-08-24)"}
for name in SMOOTH:
    RESULT[name + " (1.0.9)"] = dict(PREVIOUS[name], source="2026-09-30")

## Results summary

In [8]:
def metric_table(column, label):
    names = list(RESULT)
    values = {n: RESULT[n][column] for n in names}
    frame = pd.DataFrame({
        "Method": names,
        "Source": [RESULT[n]["source"] for n in names],
        "Min": [np.nanmin(values[n]) if values[n] is not None else np.nan for n in names],
        "Q1": [np.nanquantile(values[n], 0.25) if values[n] is not None else np.nan for n in names],
        "Med": [np.nanmedian(values[n]) if values[n] is not None else np.nan for n in names],
        "Q3": [np.nanquantile(values[n], 0.75) if values[n] is not None else np.nan for n in names],
        "Max": [np.nanmax(values[n]) if values[n] is not None else np.nan for n in names],
        "Mean": [np.nanmean(values[n]) if values[n] is not None else np.nan for n in names],
        "Failed": [int(np.sum(np.isnan(values[n]))) if values[n] is not None else np.nan
                   for n in names],
    })
    print("\n" + "=" * 110 + f"\n{label}, all {len(datasets)} series (lower = better)\n" + "=" * 110)
    print(frame.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return frame


summary_rmsse = metric_table("rmsse", "RMSSE")


RMSSE, all 5315 series (lower = better)
                  Method                 Source    Min     Q1    Med     Q3     Max   Mean  Failed
        AutoMSARIMA Back             2026-10-01 0.0246 0.6972 1.1867 2.3367 51.6160 2.0044       0
         AutoMSARIMA Opt             2026-10-01 0.0383 0.7022 1.1822 2.3515 51.6160 2.0085       0
 statsforecast AutoARIMA csv (SAME: 2026-08-24) 0.0246 0.6826 1.1849 2.3533 51.6160 1.9809       0
        skforecast Arima csv (SAME: 2026-08-24) 0.0246 0.6927 1.1914 2.3703 51.6160 1.9971       0
AutoMSARIMA Back (1.0.9)             2026-09-30 0.0246 0.7024 1.2021 2.3767 51.6160 2.0343       0
 AutoMSARIMA Opt (1.0.9)             2026-09-30 0.0246 0.7186 1.2155 2.4329 51.6160 2.0323       0


In [9]:
summary_same = metric_table("same", "SAME")


SAME, all 5315 series (lower = better)
                  Method                 Source    Min     Q1    Med     Q3      Max   Mean  Failed
        AutoMSARIMA Back             2026-10-01 0.0000 0.4148 1.0317 2.4644  54.7914 2.0541  0.0000
         AutoMSARIMA Opt             2026-10-01 0.0000 0.4064 1.0226 2.4786  54.7915 2.0567  0.0000
 statsforecast AutoARIMA csv (SAME: 2026-08-24) 0.0346 0.2638 0.5284 1.2769  72.6445 1.3566  0.0000
        skforecast Arima csv (SAME: 2026-08-24) 0.0270 0.3798 0.6931 1.8453 106.6317 2.0471  0.0000
AutoMSARIMA Back (1.0.9)             2026-09-30    NaN    NaN    NaN    NaN      NaN    NaN     NaN
 AutoMSARIMA Opt (1.0.9)             2026-09-30    NaN    NaN    NaN    NaN      NaN    NaN     NaN


In [10]:
summary_pinball = metric_table("pinball", "Scaled pinball (mean over 99 levels)")


Scaled pinball (mean over 99 levels), all 5315 series (lower = better)
                  Method                 Source    Min     Q1    Med     Q3     Max   Mean  Failed
        AutoMSARIMA Back             2026-10-01 0.0147 0.3034 0.4953 0.9586 26.4556 0.8990  0.0000
         AutoMSARIMA Opt             2026-10-01 0.0122 0.3026 0.4924 0.9575 26.4556 0.9006  0.0000
 statsforecast AutoARIMA csv (SAME: 2026-08-24) 0.0163 0.2932 0.4915 0.9611 25.6038 0.8883  0.0000
        skforecast Arima csv (SAME: 2026-08-24) 0.0161 0.2954 0.4954 0.9724 25.6295 0.9000  0.0000
AutoMSARIMA Back (1.0.9)             2026-09-30    NaN    NaN    NaN    NaN     NaN    NaN     NaN
 AutoMSARIMA Opt (1.0.9)             2026-09-30    NaN    NaN    NaN    NaN     NaN    NaN     NaN


## Change against the previous smooth run

The same series, before (smooth 1.0.9) and after (smooth 1.1.1). `Better`/`Worse` count the series where the metric went down/up.

In [11]:
def compare(column):
    rows = []
    for name in SMOOTH:
        before, after = PREVIOUS[name][column], RESULT[name][column]
        row = {"Method": name, "Metric": column.upper()}
        for tag, v in (("before", before), ("after", after)):
            row[f"Mean {tag}"] = np.nanmean(v) if v is not None else np.nan
            row[f"Med {tag}"] = np.nanmedian(v) if v is not None else np.nan
        if before is not None:
            d = after - before
            row["Better"], row["Worse"] = int(np.sum(d < -1e-8)), int(np.sum(d > 1e-8))
        rows.append(row)
    return rows


rows = [r for column in ("rmsse", "same", "pinball") for r in compare(column)]
print(f"before = {PREVIOUS_LABEL}   after = {DATE} (smooth {pkg_version('smooth')})")
change = pd.DataFrame(rows)
print(change.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

before = 2026-09-30 (smooth 1.0.9)   after = 2026-10-01 (smooth 1.1.1)
          Method  Metric  Mean before  Med before  Mean after  Med after    Better     Worse
AutoMSARIMA Back   RMSSE       2.0343      1.2021      2.0044     1.1867 2300.0000 2050.0000
 AutoMSARIMA Opt   RMSSE       2.0323      1.2155      2.0085     1.1822 2290.0000 2147.0000
AutoMSARIMA Back    SAME          NaN         NaN      2.0541     1.0317       NaN       NaN
 AutoMSARIMA Opt    SAME          NaN         NaN      2.0567     1.0226       NaN       NaN
AutoMSARIMA Back PINBALL          NaN         NaN      0.8990     0.4953       NaN       NaN
 AutoMSARIMA Opt PINBALL          NaN         NaN      0.9006     0.4924       NaN       NaN
